# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM-self-improving-ai-agents-course/blob/main/hands_on/session_05_HANDS_ON_MCP_server_Rube.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About this Notebook

## What to do first (read this now)

⏱ **Hands-on time: 15 minutes**

During the live session, **do not try to run everything** in this notebook.

**What you should do during the course:**

1. Scroll to the **Hands-on** section
2. Adjust:
   - `USE_CASE`
   - optionally `NUM_SCENARIOS` and `MAX_TURNS`
3. Run the notebook
4. Inspect the generated scenarios and tool workflows

That’s it.

---

## What this notebook is really about (read later)

This notebook introduces **large tool ecosystems** and how agents reason over **large, heterogeneous tool sets**.

Instead of hard coding every integration, the notebook searches Composio for tools that match a natural language use case and exposes those tools to the agent.

The focus is not execution performance, but **tool discovery, planning, authentication, and orchestration**.

---

## The core idea you are practicing

This notebook shows how agents move from:

* “I have one tool” <br>
→ to
* “I have hundreds of tools, how do I decide?”

The important concepts are:

* **Tool discovery** via Composio
* **Tool selection** based on a natural language use case
* **Workflow planning** before execution
* **Connection management** for authenticated tools
* **Multi turn tool reasoning**

The scenario generator (`generate_scenarios`) is the bridge between:
* Composio meta-tool schemas
* and realistic agent tasks

---

## What the TODO actually controls

The TODO section controls the **agent’s intent**.

By changing `USE_CASE`, you change:
* which tools are discovered
* how the agent frames the problem
* what kind of workflow is produced

By changing:
* `NUM_SCENARIOS`
* `MAX_TURNS`

you control the **breadth vs depth** of exploration.

This is exactly how you would prototype real world agent behavior before training.

---

## One takeaway

If you remember one thing:

**Modern agents don’t just call tools.  
They discover, plan, authenticate, and compose tools dynamically.**

Composio is one infrastructure layer that makes that possible across hundreds of app toolkits.


This notebook is for the *Hands-on* for Session 5 for Develop Self-Improving AI Agents with Reinforcement Learning Live Event with O'Reilly Media by
[Nicole Koenigstein](https://www.linkedin.com/in/nicole-koenigstein/).

# Timer

In [1]:
SET_TIMER = False  # False, True, or minutes as a number

import requests, types
url = "https://raw.githubusercontent.com/Nicolepcx/ORM-self-improving-ai-agents-course/main/timer.py"

timer = types.ModuleType("timer")
exec(requests.get(url).text, timer.__dict__)

timer.start_exam_timer(enabled=SET_TIMER, minutes=15, warn_minutes=5)

## Installation


In [2]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install openpipe-art[backend]==0.4.11 tenacity composio composio_openai openai --prerelease allow --no-cache-dir
else:
    try:
        import numpy
        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    get_vllm, get_triton = (
        ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    )
    !uv pip install --upgrade \
        openpipe-art[backend]==0.4.11 tenacity composio composio_openai openai pillow==11.3.0 protobuf==5.29.5 {get_vllm} {get_numpy} --prerelease allow --no-cache-dir
    !uv pip install -qqq {get_triton}


# Set API Keys

<font color="red" size="5">
<b>Attention for the Notebook to work </b>
</font>
<br>

you need an `OPENROUTER_API_KEY`! [get your key here](https://openrouter.ai/) and a `COMPOSIO_API_KEY` from [Composio](https://platform.composio.dev/).

In [3]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
COMPOSIO_API_KEY = os.getenv("COMPOSIO_API_KEY")

# Initialize variables that might be used later
composio_scenarios = []



# Imports

In [4]:
import json
import random
import time
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

import art
from art.trajectories import Trajectory, Choice, TrajectoryGroup
from art.mcp import generate_scenarios
from art.mcp.generate_scenarios import preview_scenarios
from art.utils.logging import info, ok, step, warn, err

from openai import OpenAI
from composio import Composio
from composio_openai import OpenAIProvider





## What are Agent Trajectories?

In ART, a **trajectory** represents a complete interaction sequence between an agent and its environment:

- **Messages**: User inputs, system prompts, assistant responses
- **Tool Calls**: Function invocations with arguments
- **Tool Results**: Responses from tool executions
- **Rewards**: Scores from RULER or other evaluators
- **Metrics**: Metadata about the interaction (turns, success, etc.)

Multiple trajectories are collected and grouped, then RULER performs **relative ranking** to determine which trajectories are better, enabling the model to learn from comparisons rather than absolute scores.


### Integrating Better Tools with Composio

**Why multiple tools matter:**
- Different app toolkits have different strengths (email, calendar, GitHub, Slack, web search, etc.)
- Combining tools enables more realistic workflows than a single API call
- A capable agent should discover, plan, authenticate, and execute across multiple services

**Composio flow in this notebook:**
- Use `composio.tools.get(...)` to fetch tools matching `USE_CASE`
- Pass those tools to the OpenAI-compatible tool-calling loop
- Let Composio execute selected tool calls through its provider
- Use generated scenarios to explore different workflows and difficulty levels

**Think about:**
- Which toolkits are relevant for your use case?
- How should the agent decide which tool to use for different task types?
- What connection or authentication steps are needed before execution?


In [5]:
# Composio Setup
# Composio provides meta tools for discovering, authenticating, and executing tools.
# Set COMPOSIO_API_KEY in your environment or .env file before running the examples.

In [6]:
# @title Composio Setup

# Composio Configuration
# 1. Create an API key at https://platform.composio.dev/
# 2. Set it in your environment or .env file as COMPOSIO_API_KEY
# 3. Pick a stable user_id for this notebook run

COMPOSIO_USER_ID = os.getenv("COMPOSIO_USER_ID", "course-user@example.com")
COMPOSIO_API_KEY = os.getenv("COMPOSIO_API_KEY") or os.environ.get("COMPOSIO_API_KEY", "")
COMPOSIO_TOOL_LIMIT = 10
COMPOSIO_FALLBACK_TOOLKITS = ["GMAIL", "OUTLOOK", "SENDGRID"]
COMPOSIO_FALLBACK_TOOLS = [
    "GMAIL_SEND_EMAIL",
    "GMAIL_CREATE_EMAIL_DRAFT",
    "GMAIL_SEND_DRAFT",
    "OUTLOOK_SEND_EMAIL",
    "SENDGRID_SEND_EMAIL_WITH_TWILIO_SEND_GRID",
]

if not COMPOSIO_API_KEY:
    warn("COMPOSIO_API_KEY not set. Please:")
    warn("1. Visit https://platform.composio.dev/")
    warn("2. Create an API key")
    warn("3. Set it in your .env file as COMPOSIO_API_KEY or export COMPOSIO_API_KEY=your_key")
    composio = None
else:
    # This notebook uses the direct tools.get(...) API from the Composio quick start.
    composio = Composio(api_key=COMPOSIO_API_KEY, provider=OpenAIProvider())
    ok("Composio configured")
    print(f"User ID: {COMPOSIO_USER_ID}")
    print("API key: [configured]")


[11:41:35] OK    Composio configured
User ID: course-user@example.com
API key: [configured]


# Hands-on

<font color="red" size="10">
<b>TODO: </b>
</font>
<br>
<font color="black" size="5">
<b>Try out to generate different scenarios by adjusting  <code>USE_CASE</code> you can also experiment with <code>NUM_SCENARIOS</code> or <code>MAX_TURNS</code>.</b>
</font>



In [7]:
NUM_SCENARIOS = 5 # Small number for demo
MAX_TURNS = 5 # Small number for demo
USE_CASE = "I want to search for tools that can help me send an email. Show me what tools are available and what the workflow would look like."

In [8]:
# @title Composio Helper Functions


def _tool_function(tool: Any) -> Dict[str, Any]:
    """Return the function payload for dict or SDK model tools."""
    if hasattr(tool, "model_dump"):
        tool = tool.model_dump()
    elif hasattr(tool, "dict"):
        tool = tool.dict()

    if not isinstance(tool, dict):
        return {}
    if "function" in tool and isinstance(tool["function"], dict):
        return tool["function"]
    if tool.get("type") == "function" and "name" in tool:
        # Some providers use the Responses API shape: {type, name, description, parameters}.
        return tool
    return tool


def _clean_json_schema(schema: Any) -> Dict[str, Any]:
    """Keep tool parameters in the JSON Schema subset accepted by Chat Completions."""
    if not isinstance(schema, dict):
        return {"type": "object", "properties": {}}

    allowed = {
        "type", "properties", "required", "description", "enum", "items", "anyOf",
        "oneOf", "allOf", "default", "additionalProperties", "format", "title",
        "minimum", "maximum", "minLength", "maxLength", "minItems", "maxItems",
    }
    cleaned = {}
    for key, value in schema.items():
        if key not in allowed:
            continue
        if key == "properties" and isinstance(value, dict):
            cleaned[key] = {name: _clean_json_schema(prop) for name, prop in value.items()}
        elif key in {"items", "additionalProperties"} and isinstance(value, dict):
            cleaned[key] = _clean_json_schema(value)
        elif key in {"anyOf", "oneOf", "allOf"} and isinstance(value, list):
            cleaned[key] = [_clean_json_schema(item) for item in value]
        else:
            cleaned[key] = value

    cleaned.setdefault("type", "object")
    if cleaned.get("type") == "object":
        cleaned.setdefault("properties", {})
    return cleaned


def composio_tool_to_art_info(tool: Any) -> Dict[str, Any]:
    """Convert a Composio/OpenAI tool into the schema shape expected by ART."""
    fn = _tool_function(tool)
    return {
        "name": fn.get("name", "UNKNOWN_TOOL"),
        "description": fn.get("description", ""),
        "parameters": _clean_json_schema(fn.get("parameters", {"type": "object", "properties": {}})),
    }


def composio_tool_to_openai_chat_tool(tool: Any) -> Dict[str, Any]:
    """Normalize Composio tools to Chat Completions/OpenRouter's strict tool shape."""
    fn = composio_tool_to_art_info(tool)
    return {
        "type": "function",
        "function": {
            "name": fn["name"],
            "description": fn["description"],
            "parameters": fn["parameters"],
        },
    }


def _short_json(value: Any, max_chars: int = 500) -> str:
    text = json.dumps(value, indent=2, default=str) if not isinstance(value, str) else value
    return text if len(text) <= max_chars else text[:max_chars] + "... [truncated]"


def _get_composio_tools(**kwargs):
    """Call tools.get across SDK versions that differ on user_id placement."""
    try:
        return composio.tools.get(user_id=COMPOSIO_USER_ID, **kwargs)
    except TypeError:
        return composio.tools.get(COMPOSIO_USER_ID, **kwargs)


def get_composio_tools_for_use_case(use_case: str, limit: int = COMPOSIO_TOOL_LIMIT):
    """Fetch Composio tools, trying fallbacks when semantic search returns no tools."""
    if composio is None:
        warn("Composio not configured - skipping Composio examples")
        return []

    lookup_attempts = [
        ("semantic search", {"search": use_case, "limit": limit}),
        ("semantic search, latest versions", {"search": use_case, "limit": limit, "toolkit_versions": "latest"}),
        ("exact email tool slugs", {"tools": COMPOSIO_FALLBACK_TOOLS}),
        ("exact email tool slugs, latest versions", {"tools": COMPOSIO_FALLBACK_TOOLS, "toolkit_versions": "latest"}),
        ("email toolkits", {"toolkits": COMPOSIO_FALLBACK_TOOLKITS, "limit": limit}),
        ("email toolkits lowercase", {"toolkits": [t.lower() for t in COMPOSIO_FALLBACK_TOOLKITS], "limit": limit}),
        ("email toolkits, latest versions", {"toolkits": COMPOSIO_FALLBACK_TOOLKITS, "limit": limit, "toolkit_versions": "latest"}),
    ]

    last_error = None
    for label, kwargs in lookup_attempts:
        try:
            tools = _get_composio_tools(**kwargs)
        except TypeError as e:
            last_error = e
            continue
        except Exception as e:
            last_error = e
            warn(f"Composio lookup failed for {label}: {e}")
            continue

        if tools:
            ok(f"Loaded {len(tools)} Composio tool(s) via {label}")
            return tools

        info(f"No tools returned via {label}; trying next lookup strategy.")

    if last_error:
        warn(f"No Composio tools found. Last lookup error: {last_error}")
    else:
        warn("No Composio tools found after all lookup strategies.")
    return []


async def list_composio_tools_and_resources():
    """Fetch Composio tools used by the agent for the current USE_CASE."""
    tools = get_composio_tools_for_use_case(USE_CASE)
    resources = []
    return tools, resources


if composio is not None:
    ok("Composio helpers configured")
else:
    warn("Composio not configured - skipping Composio examples")


[11:41:35] OK    Composio helpers configured


In [9]:
# @title OpenRouter + Composio: Comprehensive Workflow Example

async def list_composio_tools_example(tools: List[Any]):
    """Example: List the Composio tools selected for the current use case."""
    tool_infos = [composio_tool_to_art_info(tool) for tool in tools]
    print(f"📋 Found {len(tool_infos)} Composio tools for this use case:\n")

    tools_by_toolkit = {}
    for tool in tool_infos:
        toolkit = tool["name"].split("_")[0] if "_" in tool["name"] else "OTHER"
        tools_by_toolkit.setdefault(toolkit, []).append(tool)

    for toolkit, toolkit_tools in tools_by_toolkit.items():
        print(f"  {toolkit.title()}:")
        for tool in toolkit_tools:
            print(f"    - {tool['name']}")
            if tool["description"]:
                desc = tool["description"].split("\n")[0][:100]
                print(f"      {desc}...")
        print()

    return tool_infos


async def composio_workflow_example(use_case: str):
    """Example: Search Composio tools for a use case, then let the model use them."""
    tools = get_composio_tools_for_use_case(use_case)
    if not tools:
        warn("No Composio tools available")
        return ""

    # Composio may return provider-specific tool shapes. OpenRouter's Chat
    # Completions endpoint expects the strict {type: function, function: ...} shape.
    chat_tools = [composio_tool_to_openai_chat_tool(tool) for tool in tools]
    chat_tools = [tool for tool in chat_tools if tool["function"]["name"] != "UNKNOWN_TOOL"]
    if not chat_tools:
        warn("No Chat Completions-compatible Composio tools available")
        return ""

    llm = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

    tool_names = [tool["function"]["name"] for tool in chat_tools]
    system_prompt = f"""You are an AI assistant with access to Composio tools for workflow automation.

The notebook has already searched Composio for tools relevant to the user's use case and exposed only those tools to you.

Available tool names:
{json.dumps(tool_names, indent=2)}

Workflow pattern:
1. Explain which available tools match the user's goal.
2. If a tool requires account authentication and execution fails or is not possible, explain which app connection is needed.
3. Only execute a tool when the user has provided enough concrete inputs.
4. If the user is only asking to discover tools or see a workflow, summarize the tools and recommended workflow instead of performing a real-world action.
"""

    messages: list[dict[str, Any]] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": use_case},
    ]

    print(f"🚀 Starting Composio workflow for: {use_case}\n")

    for turn in range(MAX_TURNS):
        response = llm.chat.completions.create(
            model="openai/o4-mini",
            messages=messages,
            tools=chat_tools,
            tool_choice="auto" if chat_tools else None,
        )

        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if msg.content:
            print(f"💬 Assistant: {msg.content}\n")

        if not msg.tool_calls:
            break

        print(f"🔧 Tool calls ({len(msg.tool_calls)}):")
        for tc in msg.tool_calls:
            tool_args = json.loads(tc.function.arguments or "{}")
            print(f"  - {tc.function.name}")
            print(f"    Args: {_short_json(tool_args, max_chars=300)}")

        try:
            results = composio.provider.handle_tool_calls(
                response=response,
                user_id=COMPOSIO_USER_ID,
            )
        except Exception as e:
            # For discovery/planning demos, keep the notebook moving even if the
            # account is not connected or the SDK cannot execute normalized calls.
            results = [
                {
                    "successful": False,
                    "error": str(e),
                    "note": "Tool call was generated, but execution was skipped or failed. Connect the required app account before real execution.",
                }
                for _ in msg.tool_calls
            ]

        for tc, result in zip(msg.tool_calls, results):
            print(f"    Result: {_short_json(result, max_chars=500)}\n")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, default=str),
            })

    return messages[-1].get("content", "") if isinstance(messages[-1], dict) else ""


# Example 1: List available Composio tools for this use case
if composio is not None:
    print("=" * 60)
    print("Example 1: Listing Composio Tools")
    print("=" * 60)
    try:
        tools, _ = await list_composio_tools_and_resources()
        tool_infos = await list_composio_tools_example(tools)
        ok(f"Successfully listed {len(tool_infos)} tools")
    except Exception as e:
        warn(f"Failed to list tools: {e}")
else:
    warn("Composio not configured. Please set COMPOSIO_API_KEY.")


# Example 2: Complete workflow (requires OpenRouter API key)
if composio is not None and OPENROUTER_API_KEY:
    print("\n" + "=" * 60)
    print("Example 2: Composio Workflow - Search for Tools")
    print("=" * 60)

    try:
        result = await composio_workflow_example(use_case=USE_CASE)
        print("\n✅ Workflow completed!")
    except Exception as e:
        warn(f"Workflow failed: {e}")
else:
    info("\n💡 To run the full workflow example:")
    info("   1. Set COMPOSIO_API_KEY in your .env file")
    info("   2. Set OPENROUTER_API_KEY in your .env file")
    info("   3. Re-run this cell")


Example 1: Listing Composio Tools
[11:41:36] INFO  No tools returned via semantic search; trying next lookup strategy.
[11:41:37] OK    Loaded 4 Composio tool(s) via exact email tool slugs
📋 Found 4 Composio tools for this use case:

  Gmail:
    - GMAIL_CREATE_EMAIL_DRAFT
      Creates a gmail email draft, supporting to/cc/bcc, subject, plain/html body (ensure `is html=true` f...
    - GMAIL_SEND_DRAFT
      Sends the specified, existing draft to the recipients in the to, cc, and bcc headers....
    - GMAIL_SEND_EMAIL
      Sends an email via gmail api using the authenticated user's google profile display name, requiring `...

  Sendgrid:
    - SENDGRID_SEND_EMAIL_WITH_TWILIO_SEND_GRID
      The mail send operation uses sendgrid's v3 api to send emails. visit the provided link for an overvi...

[11:41:37] OK    Successfully listed 4 tools

Example 2: Composio Workflow - Search for Tools
[11:41:37] INFO  No tools returned via semantic search; trying next lookup strategy.
[11:41:38] OK 

In [10]:
# @title Generate Scenarios with Composio Tools

# Simple example: Generate scenarios for one Composio tool selected by USE_CASE

if composio is not None and OPENROUTER_API_KEY:
    # Step 1: Get available tools from Composio's semantic tool search
    tools_result, resources_result = await list_composio_tools_and_resources()
    tool_infos = [composio_tool_to_art_info(tool) for tool in tools_result]

    # Step 2: Choose the first matching tool for a focused demo
    selected_tool = tool_infos[0] if tool_infos else None

    if selected_tool:
        selected_tool_name = selected_tool["name"]
        info(f"Selected tool: {selected_tool_name}")

        # Step 3: Generate scenarios (similar to the other tool examples)
        try:
            scenario_collection = await generate_scenarios(
                tools=[selected_tool],
                resources=[],
                num_scenarios=NUM_SCENARIOS,
                show_preview=True,
                generator_model="openai/o4-mini",
                generator_api_key=OPENROUTER_API_KEY,
            )

            scenarios = [{"task": s.task, "difficulty": s.difficulty} for s in scenario_collection.scenarios]
            composio_scenarios.extend(scenarios)
            ok(f"Generated {len(scenarios)} scenarios for {selected_tool_name}")

            info("Sample scenarios:")
            preview_scenarios(scenarios, n=min(3, len(scenarios)))

        except Exception as e:
            warn(f"Scenario generation failed: {e}")
    else:
        warn("No matching Composio tools found for this USE_CASE after all fallback lookups.")
        warn("If you do not see lookup-strategy messages above, rerun the Composio Helper Functions cell first.")
else:
    warn("Composio or OpenRouter not configured. Please set COMPOSIO_API_KEY and OPENROUTER_API_KEY.")

[11:41:47] INFO  No tools returned via semantic search; trying next lookup strategy.
[11:41:47] OK    Loaded 4 Composio tool(s) via exact email tool slugs
[11:41:47] INFO  Selected tool: GMAIL_CREATE_EMAIL_DRAFT
[11:41:47] OK    Using model: openai/o4-mini
[11:41:47] INFO  Available: 1 tool(s), 0 resource(s).
[11:41:47] STEP  Preparing prompt & JSON schema &
[11:41:47] STEP  Calling model: openai/o4-mini &
[11:41:54] OK    Model responded in 6.71s.
[11:41:54] INFO  Raw content length: 1586 chars.
[11:41:54] OK    Parsed 5 scenario(s) successfully.
[11:41:54] INFO  Difficulty distribution:
   1/5:   1  █
   2/5:   1  █
   3/5:   1  █
   4/5:   1  █
   5/5:   1  █
   1. Draft a brief plain‐text email to a colleague inviting them to a one‐hour project kickoff meeting next Tuesday at 10 AM.…  (difficulty 1/5)
   2. Compose an email to your manager with an update on the quarterly sales figures, cc’ing the finance team and bcc’ing your…  (difficulty 2/5)
   3. Reply in an existing email thre

In [11]:
# @title Collect Tools from Composio
search_tools = []
all_tools = {}

# Collect Composio tools matching the current USE_CASE
if composio is not None:
    try:
        tools_result, resources_result = await list_composio_tools_and_resources()
        composio_tools = []
        for tool in tools_result:
            tool_dict = composio_tool_to_art_info(tool)
            tool_dict["source"] = "composio"
            search_tools.append(tool_dict)
            composio_tools.append(tool_dict)
        all_tools["composio"] = composio_tools
        if composio_tools:
            ok(f"Collected {len(composio_tools)} tool(s) from Composio")
    except Exception as e:
        warn(f"Failed to collect Composio tools: {e}")
        all_tools["composio"] = []

if search_tools:
    info(f"Total tools collected: {len(search_tools)} from {len(all_tools)} source(s)")
else:
    warn("No tools collected after all Composio lookup strategies.")
    warn("If you do not see lookup-strategy messages above, rerun the Composio Helper Functions cell first.")

[11:41:55] INFO  No tools returned via semantic search; trying next lookup strategy.
[11:41:55] OK    Loaded 4 Composio tool(s) via exact email tool slugs
[11:41:55] OK    Collected 4 tool(s) from Composio
[11:41:55] INFO  Total tools collected: 4 from 1 source(s)


In [12]:
# @title Generate Scenarios with Enhanced Tool Set

async def generate_scenarios_with_enhanced_tools(search_tools: List[Dict], num_scenarios: int = 15):
    """
    Generate scenarios that leverage the enhanced tool set.
    These scenarios should encourage the agent to use multiple tools effectively.
    """
    if not OPENROUTER_API_KEY:
        warn("OPENROUTER_API_KEY required for scenario generation")
        return []

    # Organize tools by source for scenario generation
    tools_by_source = {}
    for tool in search_tools:
        source = tool["source"]
        if source not in tools_by_source:
            tools_by_source[source] = []
        tools_by_source[source].append({
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": tool.get("parameters", {})
        })

    info(f"Generating {num_scenarios} scenarios with enhanced tool set...")
    info(f"Tools available from: {', '.join(tools_by_source.keys())}")

    # Combine all tools for scenario generation
    all_tools_flat = [
        {
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": tool.get("parameters", {})
        }
        for tool in search_tools
    ]

    try:
        scenario_collection = await generate_scenarios(
            tools=all_tools_flat,
            resources=[],  # Can add resources if available
            num_scenarios=num_scenarios,
            show_preview=False,
            generator_model="openai/o4-mini",
            generator_api_key=OPENROUTER_API_KEY,
        )

        enhanced_scenarios = [
            {
                "task": s.task,
                "difficulty": s.difficulty,
                "tools_available": len(all_tools_flat)
            }
            for s in scenario_collection.scenarios
        ]

        ok(f"Generated {len(enhanced_scenarios)} enhanced scenarios")

        info("\nSample enhanced scenarios (encouraging multi-tool usage):")
        preview_scenarios(enhanced_scenarios, n=min(5, len(enhanced_scenarios)))

        return enhanced_scenarios

    except Exception as e:
        warn(f"Scenario generation failed: {e}")
        return []

# Generate enhanced scenarios
if search_tools and OPENROUTER_API_KEY:
    enhanced_scenarios = await generate_scenarios_with_enhanced_tools(
        search_tools,
        num_scenarios=15
    )


[11:41:55] INFO  Generating 15 scenarios with enhanced tool set...
[11:41:55] INFO  Tools available from: composio
[11:41:55] OK    Using model: openai/o4-mini
[11:41:55] INFO  Available: 4 tool(s), 0 resource(s).
[11:41:55] STEP  Preparing prompt & JSON schema &
[11:41:55] STEP  Calling model: openai/o4-mini &
[11:42:11] OK    Model responded in 16.06s.
[11:42:11] INFO  Raw content length: 4151 chars.
[11:42:11] OK    Parsed 15 scenario(s) successfully.
[11:42:11] INFO  Difficulty distribution:
   1/5:   2  ██
   2/5:   4  ████
   3/5:   3  ███
   4/5:   3  ███
   5/5:   3  ███
[11:42:11] OK    Generated 15 scenarios in 16.13s total.
[11:42:11] OK    Generated 15 enhanced scenarios
[11:42:11] INFO  
Sample enhanced scenarios (encouraging multi-tool usage):
   1. Send a simple plain-text email to a colleague with the subject "Meeting Agenda" outlining the topics for tomorrow’s meet&  (difficulty 1/5)
   2. Create a draft of a project proposal email in HTML format with a PDF attachment,

### Summary: Improving Your Agent

**What we've covered:**

1. **More Comparisons** ✅
   - Generate 5-10+ trajectories per scenario (not just 2-3)
   - Use RULER to rank trajectories relatively
   - Vary strategies (different models, prompts, tool usage patterns)
   - More comparisons = better learning signal for the model

2. **Better Tools** ✅
   - Integrate Composio as a large dynamic tool ecosystem
   - Select complementary tools and toolkits for each use case
   - Generate scenarios that encourage multi-tool usage
   - Train agent to intelligently combine results


**Remember**: RULER learns what "good" means from your specific tools and scenarios - no labeled data required!


### Additional Resources

- **ART Documentation**: https://art.openpipe.ai
- **RULER Guide**: https://art.openpipe.ai/fundamentals/ruler
- **Composio SDK**: https://github.com/composiohq/composio
- **Composio Docs**: https://docs.composio.dev/
